# Long-Sequence CurvBias Ablation (v6)

Targeted ablation of CurvBias position encoding with three fixes for long-sequence scaling,
tested at T=3072 on H100.

## Fixes applied (vs v5)

| Fix | Problem | Solution |
|-----|---------|----------|
| **A** | `delta = tanh(·) * pi/T` — per-step gradient vanishes as T grows | Decoupled: free per-step `tanh(delta)` + `pi * tanh(cumsum / C)` caps total drift at ±pi. C is learnable per head. |
| **B** | `cdist` on unwrapped angles — wrong metric (unbounded, ignores torus topology) | Circular distance: `sum(1 - cos(θ_t - θ_s))` via GEMM on existing cos/sin |
| **G** | Full T×T bias computed — wasted compute on causally-masked upper triangle | Causal masking on bias before scaling |

## Methods

```
RoPE:       θ[t] = t · freq                                    # fixed, abelian
CurvBias:   CDRoPE + Σ_d(1 - cos(θ_t,d − θ_s,d)) as bias     # circular curvature (FIXED)
  where     θ[t] = t·freq + π·tanh(cumsum(tanh(f(x))) / C)    # soft-capped drift, C learnable
```

## Models

| # | Model | Position Encoding | Attention | What it tests |
|---|-------|-------------------|-----------|---------------|
| 1 | GPT+RoPE | Fixed rotary | Softmax | Baseline |
| 2 | GPT+CurvBias | CDRoPE + circular bias | Softmax | **Novel**: gauge-theory bias (fixed) |
| 3 | GLA+RoPE | Fixed rotary | Linear | Baseline |
| 4 | GLA+CurvBias | CDRoPE + circular bias | Linear | **Novel**: curvature on linear (fixed) |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from tqdm.auto import tqdm
import math
import time
import os

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name()}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

try:
    from datasets import load_dataset
    print("Loading WikiText-103...")
    ds = load_dataset("wikitext", "wikitext-103-raw-v1")
    train_text = "\n".join(ds["train"]["text"])
    val_text = "\n".join(ds["validation"]["text"])
except ImportError:
    raise ImportError("pip install datasets")

try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    vocab_size = enc.n_vocab
except ImportError:
    from transformers import GPT2TokenizerFast
    enc = GPT2TokenizerFast.from_pretrained("gpt2")
    vocab_size = enc.vocab_size

def tokenize(text):
    if hasattr(enc, 'encode_ordinary'):
        return enc.encode_ordinary(text)
    if hasattr(enc, 'encode'):
        return enc.encode(text)
    return enc(text)['input_ids']

print("Tokenizing...")
train_ids = torch.tensor(tokenize(train_text), dtype=torch.long)
val_ids = torch.tensor(tokenize(val_text), dtype=torch.long)
print(f"Train: {len(train_ids):,} tokens, Val: {len(val_ids):,} tokens, Vocab: {vocab_size:,}")

Device: cuda
  GPU: NVIDIA H100 80GB HBM3
  Memory: 85.0 GB
Loading WikiText-103...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizing...
Train: 118,496,151 tokens, Val: 248,461 tokens, Vocab: 50,257


In [2]:
@dataclass
class AblationConfig:
    d_model: int = 512
    n_heads: int = 8
    head_dim: int = 64
    conv_kernel: int = 4           # short causal conv for GLA

    ffn_mult: int = 4
    n_blocks: int = 8
    vocab_size: int = 50257
    max_seq_len: int = 3072
    dropout: float = 0.1

    learning_rate: float = 5e-4
    min_lr: float = 5e-5
    warmup_steps: int = 1000
    lr_hold_steps: int = 7000
    batch_size: int = 4
    seq_len: int = 3072
    max_steps: int = 40000
    eval_interval: int = 5000
    eval_steps: int = 500

cfg = AblationConfig(vocab_size=vocab_size)
print(f"Config: d={cfg.d_model} heads={cfg.n_heads} hd={cfg.head_dim}")
print(f"  Blocks: {cfg.n_blocks}, seq_len: {cfg.seq_len}, batch: {cfg.batch_size}")
print(f"  LR: {cfg.learning_rate}, steps: {cfg.max_steps}")
print(f"  Fix A drift_cap init: sqrt({cfg.max_seq_len}) = {math.sqrt(cfg.max_seq_len):.1f} (learnable per head)")

def get_batch(data, c):
    ix = torch.randint(0, len(data) - c.seq_len - 1, (c.batch_size,))
    return torch.stack([data[i:i+c.seq_len] for i in ix]).to(device)

Config: d=512 heads=8 hd=64
  Blocks: 8, seq_len: 3072, batch: 4
  LR: 0.0005, steps: 40000
  Fix A drift_cap init: sqrt(3072) = 55.4 (learnable per head)


## Shared Utilities: Rotary Encoding

In [3]:
def apply_rotary(x, cos_t, sin_t):
    """Standard rotary embedding. x: (B, H, T, d), cos/sin: (..., d/2)."""
    d2 = x.shape[-1] // 2
    x1, x2 = x[..., :d2], x[..., d2:]
    return torch.cat([x1 * cos_t - x2 * sin_t, x1 * sin_t + x2 * cos_t], dim=-1)


def circular_curvature_bias(cos_t, sin_t, causal_mask_2d):
    """Compute circular distance bias on the angle torus (fixes B + G).

    d(t,s) = sum_d(1 - cos(theta_t,d - theta_s,d))
           = n_rot - (cos_t @ cos_s^T + sin_t @ sin_s^T)

    Bounded in [0, 2*n_rot]. Correct torus metric. Reuses cos/sin from RoPE.
    Uses efficient GEMM instead of cdist.

    Args:
        cos_t: (B, H, T, n_rot) — cos of CDRoPE angles
        sin_t: (B, H, T, n_rot) — sin of CDRoPE angles
        causal_mask_2d: (T, T) — lower-triangular mask

    Returns:
        bias: (B, H, T, T) — circular distance, causal-masked
    """
    B, H, T, n_rot = cos_t.shape

    # Reshape for batched matmul: (B*H, T, n_rot)
    cos_flat = cos_t.reshape(B * H, T, n_rot)
    sin_flat = sin_t.reshape(B * H, T, n_rot)

    # Gram matrices via GEMM (much faster than cdist on GPU)
    gram = (torch.bmm(cos_flat, cos_flat.transpose(-1, -2)) +
            torch.bmm(sin_flat, sin_flat.transpose(-1, -2)))

    # Circular distance: sum_d(1 - cos(d_theta)) = n_rot - gram
    bias = (n_rot - gram).reshape(B, H, T, T)

    # Fix G: zero upper triangle (causal-only)
    bias = bias * causal_mask_2d

    return bias


# Quick test
_cos = torch.randn(2, 8, 16, 32)
_sin = torch.randn(2, 8, 16, 32)
_mask = torch.tril(torch.ones(16, 16))
_bias = circular_curvature_bias(_cos, _sin, _mask)
print(f"circular_curvature_bias: input ({_cos.shape}) -> bias {_bias.shape}")
print(f"  Upper triangle all zero: {(_bias[:,:,0,1:] == 0).all().item()}")
print(f"  Diagonal all zero: {(_bias[:,:,range(16),range(16)] == 0).all().item()}")

circular_curvature_bias: input (torch.Size([2, 8, 16, 32])) -> bias torch.Size([2, 8, 16, 16])
  Upper triangle all zero: True
  Diagonal all zero: False


## Gated Linear Attention with RoPE / CurvBias

CurvBias mode applies three fixes for long-sequence scaling:
- **Fix A**: Decoupled per-step budget + learnable drift cap: `pi * tanh(cumsum / C)`
- **Fix B**: Circular distance via GEMM on cos/sin (bounded, correct torus metric)
- **Fix G**: Causal mask on bias (clean gradients, no wasted upper-triangle compute)

In [4]:
class GatedLinearAttention(nn.Module):
    """Gated linear attention with RoPE or CurvBias position encoding.

    rope_mode:
      'none'      — no position encoding
      'rope'      — standard RoPE (fixed per position)
      'curvbias'  — CDRoPE + circular curvature bias (v6 fixed)
    """
    def __init__(self, cfg, rope_mode='none'):
        super().__init__()
        D = cfg.d_model
        H = cfg.n_heads
        hd = cfg.head_dim
        n_rot = hd // 2
        self.rope_mode = rope_mode

        # Short causal conv
        self.conv = nn.Conv1d(D, D, kernel_size=cfg.conv_kernel,
                              padding=cfg.conv_kernel - 1, groups=D)

        # QKV + output
        self.q_proj = nn.Linear(D, H * hd)
        self.k_proj = nn.Linear(D, H * hd)
        self.v_proj = nn.Linear(D, H * hd)
        self.o_proj = nn.Linear(H * hd, D)

        # Gates
        self.alpha_proj = nn.Linear(D, H)
        nn.init.zeros_(self.alpha_proj.weight)
        nn.init.constant_(self.alpha_proj.bias, 3.0)
        self.beta_proj = nn.Linear(D, H)
        nn.init.zeros_(self.beta_proj.weight)
        nn.init.constant_(self.beta_proj.bias, 1.0)

        # Position encoding setup
        if rope_mode in ('rope', 'curvbias'):
            freqs = 1.0 / (10000.0 ** (torch.arange(0, n_rot, dtype=torch.float32) / n_rot))
            self.register_buffer('base_freqs', freqs)
        if rope_mode == 'curvbias':
            self.theta_proj = nn.Linear(D, H * n_rot)
            nn.init.zeros_(self.theta_proj.weight)
            nn.init.zeros_(self.theta_proj.bias)
            self.curv_alpha = nn.Parameter(torch.zeros(1, H, 1, 1))
            # Fix A: learnable per-head drift cap — total content drift bounded
            # at ±pi via tanh, but per-step gradients don't vanish with T
            self.drift_cap = nn.Parameter(torch.ones(1, H, 1, 1) * math.sqrt(cfg.max_seq_len))

        self.H = H
        self.hd = hd
        self.n_rot = n_rot
        self.o_drop = nn.Dropout(cfg.dropout)
        self.register_buffer('causal_mask',
            torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len)))

    def forward(self, x):
        B, T, D = x.shape
        H, hd = self.H, self.hd
        BH = B * H

        # Short causal conv
        x_conv = self.conv(x.transpose(1, 2))[:, :, :T].transpose(1, 2)

        # Project Q, K, V
        q = self.q_proj(x_conv).reshape(B, T, H, hd).permute(0, 2, 1, 3)
        k = self.k_proj(x_conv).reshape(B, T, H, hd).permute(0, 2, 1, 3)
        v = self.v_proj(x_conv).reshape(B, T, H, hd).permute(0, 2, 1, 3)
        k = F.normalize(k, dim=-1)

        # === Position encoding ===
        curv_bias = None
        if self.rope_mode == 'rope':
            positions = torch.arange(T, device=x.device, dtype=x.dtype)
            theta = (positions.unsqueeze(-1) * self.base_freqs).reshape(1, 1, T, self.n_rot)
            theta = theta.expand(B, H, T, self.n_rot)
            cos_t, sin_t = torch.cos(theta), torch.sin(theta)
            q = apply_rotary(q, cos_t, sin_t)
            k = apply_rotary(k, cos_t, sin_t)
        elif self.rope_mode == 'curvbias':
            positions = torch.arange(T, device=x.device, dtype=x.dtype)
            base_theta = (positions.unsqueeze(-1) * self.base_freqs).reshape(1, 1, T, self.n_rot)
            delta = self.theta_proj(x_conv).reshape(B, T, H, self.n_rot)
            delta = delta.permute(0, 2, 1, 3)
            # Fix A: per-step freely learned (tanh bounds to [-1,1]),
            # total drift soft-capped at ±pi via tanh(cumsum / drift_cap)
            delta = torch.tanh(delta)
            cum_delta = torch.cumsum(delta, dim=2)
            theta = base_theta + math.pi * torch.tanh(cum_delta / self.drift_cap)
            cos_t, sin_t = torch.cos(theta), torch.sin(theta)
            q = apply_rotary(q, cos_t, sin_t)
            k = apply_rotary(k, cos_t, sin_t)
            # Fix B + G: circular distance, causal-only
            curv_bias = circular_curvature_bias(cos_t, sin_t, self.causal_mask[:T, :T])
            curv_bias = self.curv_alpha * curv_bias

        # === Gated linear attention ===
        q = q.reshape(BH, T, hd)
        k = k.reshape(BH, T, hd)
        v = v.reshape(BH, T, hd)

        alpha = torch.sigmoid(self.alpha_proj(x_conv)).permute(0, 2, 1).reshape(BH, T)
        beta = torch.sigmoid(self.beta_proj(x_conv)).permute(0, 2, 1).reshape(BH, T)

        scores = torch.bmm(q, k.transpose(-1, -2)) * (hd ** -0.5)

        if curv_bias is not None:
            scores = scores + curv_bias.reshape(BH, T, T)

        pos_scores = F.softplus(scores)

        log_alpha = torch.log(alpha.clamp(min=1e-6))
        log_G = torch.cumsum(log_alpha, dim=1)
        log_decay = log_G.unsqueeze(-1) - log_G.unsqueeze(-2)
        log_decay = log_decay.masked_fill(self.causal_mask[:T, :T] == 0, float('-inf'))

        att = pos_scores * torch.exp(log_decay) * beta.unsqueeze(-2)
        output = torch.bmm(att, v)
        z = att.sum(dim=-1, keepdim=True).clamp(min=1e-6)
        output = output / z

        output = output.reshape(B, H, T, hd).permute(0, 2, 1, 3).reshape(B, T, H * hd)
        return self.o_drop(self.o_proj(output))


# Quick test
for mode in ['rope', 'curvbias']:
    _cfg = AblationConfig()
    _layer = GatedLinearAttention(_cfg, rope_mode=mode)
    _out = _layer(torch.randn(2, 16, _cfg.d_model))
    print(f"  GLA rope_mode={mode:10s}: params={sum(p.numel() for p in _layer.parameters()):,}")

  GLA rope_mode=rope      : params=1,061,392
  GLA rope_mode=curvbias  : params=1,192,736


## Softmax Attention with RoPE / CurvBias

Same CurvBias fixes (A, B, G) applied to standard causal softmax attention.

In [5]:
class SoftmaxAttention(nn.Module):
    """Causal softmax attention with RoPE or CurvBias position encoding.

    rope_mode:
      'none'      — no rotary (use external pos emb)
      'rope'      — standard RoPE (fixed per position)
      'curvbias'  — CDRoPE + circular curvature bias (v6 fixed)
    """
    def __init__(self, cfg, rope_mode='none', layer_idx=0, n_layers=8):
        super().__init__()
        D = cfg.d_model
        H = cfg.n_heads
        hd = cfg.head_dim
        n_rot = hd // 2
        self.rope_mode = rope_mode

        self.qkv = nn.Linear(D, 3 * D)
        self.proj = nn.Linear(D, D)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.proj_drop = nn.Dropout(cfg.dropout)

        if rope_mode in ('rope', 'curvbias'):
            freqs = 1.0 / (10000.0 ** (torch.arange(0, n_rot, dtype=torch.float32) / n_rot))
            self.register_buffer('base_freqs', freqs)
        if rope_mode == 'curvbias':
            self.theta_proj = nn.Linear(D, H * n_rot)
            nn.init.zeros_(self.theta_proj.weight)
            nn.init.zeros_(self.theta_proj.bias)
            self.curv_alpha = nn.Parameter(torch.zeros(1, H, 1, 1))
            # Fix A: learnable per-head drift cap
            self.drift_cap = nn.Parameter(torch.ones(1, H, 1, 1) * math.sqrt(cfg.max_seq_len))

        self.H = H
        self.hd = hd
        self.n_rot = n_rot
        self.register_buffer('causal_mask',
            torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len))
                  .view(1, 1, cfg.max_seq_len, cfg.max_seq_len))

    def forward(self, x):
        B, T, D = x.shape
        H, hd = self.H, self.hd

        qkv = self.qkv(x).reshape(B, T, 3, H, hd)
        q, k, v = qkv.unbind(2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        curv_bias = None
        if self.rope_mode == 'rope':
            positions = torch.arange(T, device=x.device, dtype=x.dtype)
            theta = (positions.unsqueeze(-1) * self.base_freqs).reshape(1, 1, T, self.n_rot)
            theta = theta.expand(B, H, T, self.n_rot)
            cos_t, sin_t = torch.cos(theta), torch.sin(theta)
            q = apply_rotary(q, cos_t, sin_t)
            k = apply_rotary(k, cos_t, sin_t)
        elif self.rope_mode == 'curvbias':
            positions = torch.arange(T, device=x.device, dtype=x.dtype)
            base_theta = (positions.unsqueeze(-1) * self.base_freqs).reshape(1, 1, T, self.n_rot)
            delta = self.theta_proj(x).reshape(B, T, H, self.n_rot).permute(0, 2, 1, 3)
            # Fix A: decoupled per-step budget + total drift cap
            delta = torch.tanh(delta)
            cum_delta = torch.cumsum(delta, dim=2)
            theta = base_theta + math.pi * torch.tanh(cum_delta / self.drift_cap)
            cos_t, sin_t = torch.cos(theta), torch.sin(theta)
            q = apply_rotary(q, cos_t, sin_t)
            k = apply_rotary(k, cos_t, sin_t)
            # Fix B + G: circular distance, causal-only
            causal_2d = self.causal_mask[0, 0, :T, :T]
            curv_bias = circular_curvature_bias(cos_t, sin_t, causal_2d)
            curv_bias = self.curv_alpha * curv_bias

        att = (q @ k.transpose(-2, -1)) * (hd ** -0.5)

        if curv_bias is not None:
            att = att + curv_bias

        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).reshape(B, T, D)
        return self.proj_drop(self.proj(y))


# Quick test
for mode in ['rope', 'curvbias']:
    _cfg = AblationConfig()
    _layer = SoftmaxAttention(_cfg, rope_mode=mode)
    _out = _layer(torch.randn(2, 16, _cfg.d_model))
    print(f"  Softmax rope_mode={mode:10s}: params={sum(p.numel() for p in _layer.parameters()):,}")

  Softmax rope_mode=rope      : params=1,050,624
  Softmax rope_mode=curvbias  : params=1,181,968


## Models

In [6]:
class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg.d_model
        self.net = nn.Sequential(
            nn.Linear(D, D * cfg.ffn_mult),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(D * cfg.ffn_mult, D),
            nn.Dropout(cfg.dropout),
        )
    def forward(self, x):
        return self.net(x)


class GLABlock(nn.Module):
    def __init__(self, cfg, rope_mode='none'):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn = GatedLinearAttention(cfg, rope_mode=rope_mode)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FFN(cfg)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class GLAModel(nn.Module):
    def __init__(self, cfg, rope_mode='none'):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = None
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([GLABlock(cfg, rope_mode) for _ in range(cfg.n_blocks)])
        self.norm_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size)

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.tok_emb(token_ids)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm_f(x))[:, :-1, :], {}


class SoftmaxBlock(nn.Module):
    def __init__(self, cfg, rope_mode='none', layer_idx=0):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn = SoftmaxAttention(cfg, rope_mode=rope_mode,
                                      layer_idx=layer_idx, n_layers=cfg.n_blocks)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FFN(cfg)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class SoftmaxModel(nn.Module):
    def __init__(self, cfg, rope_mode='none'):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = None
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([SoftmaxBlock(cfg, rope_mode, layer_idx=i)
                                     for i in range(cfg.n_blocks)])
        self.norm_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size)

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.tok_emb(token_ids)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm_f(x))[:, :-1, :], {}


def count_params(m):
    return sum(p.numel() for p in m.parameters())

# Clear any existing models
if 'models' in dir() and isinstance(models, dict):
    for name, model in models.items():
        del model
    models.clear()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

models = {}
# models['GPT+RoPE']      = SoftmaxModel(cfg, rope_mode='rope').to(device)
models['GPT+CurvBias']  = SoftmaxModel(cfg, rope_mode='curvbias').to(device)
# models['GLA+RoPE']      = GLAModel(cfg, rope_mode='rope').to(device)
models['GLA+CurvBias']  = GLAModel(cfg, rope_mode='curvbias').to(device)

print(f"\n{'Model':<18} {'Total':>10}  {'Blocks':>10}  {'Emb':>10}")
print('=' * 55)
for name, m in models.items():
    total = count_params(m)
    blk = sum(count_params(b) for b in m.blocks)
    print(f"{name:<18} {total:>10,}  {blk:>10,}  {total-blk:>10,}")


Model                   Total      Blocks         Emb
GPT+CurvBias       77,784,273  26,269,824  51,514,449
GLA+CurvBias       77,870,417  26,355,968  51,514,449


## Speed Check

In [7]:
print("Speed check (10 iterations)...")
x_test = torch.randint(0, cfg.vocab_size, (cfg.batch_size, cfg.seq_len), device=device)

for name, model in models.items():
    model.train()
    # Warmup
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(device.type == 'cuda')):
        logits, _ = model(x_test)
        loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), x_test[:, 1:].reshape(-1))
    loss.backward()
    if device.type == 'cuda': torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(10):
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(device.type == 'cuda')):
            logits, _ = model(x_test)
            loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), x_test[:, 1:].reshape(-1))
        loss.backward()
    if device.type == 'cuda': torch.cuda.synchronize()

    elapsed = (time.time() - t0) / 10
    mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == 'cuda' else 0
    print(f"  {name:<18}: {elapsed*1000:.0f}ms/step  peak_mem: {mem:.1f}GB")

    # Free GPU memory for next model
    model.cpu()
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

Speed check (10 iterations)...
  GPT+CurvBias      : 262ms/step  peak_mem: 39.1GB
  GLA+CurvBias      : 355ms/step  peak_mem: 65.8GB


## Training

In [8]:
@torch.no_grad()
def estimate_loss(model, c):
    model.eval()
    results = {}
    for name, sd in [('train', train_ids), ('val', val_ids)]:
        tot_ce, tot_ok, tot_n = 0., 0, 0
        for _ in range(c.eval_steps):
            b = get_batch(sd, c)
            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(device.type == 'cuda')):
                logits, _ = model(b)
                tgt = b[:, 1:]
                ce = F.cross_entropy(logits.reshape(-1, c.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
        n = c.eval_steps
        results[name] = {'ce': tot_ce / n, 'acc': tot_ok / tot_n}
    model.train()
    return results

import json as _json

RESULTS_DIR = 'v6_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_hist(hist, label):
    path = os.path.join(RESULTS_DIR, f'{label}.json')
    with open(path, 'w') as f:
        _json.dump(hist, f, indent=1)

def train_model(model, c, label='model'):
    opt = torch.optim.AdamW(model.parameters(), lr=c.learning_rate, weight_decay=0.05)
    mr = c.min_lr / c.learning_rate
    he = c.warmup_steps + c.lr_hold_steps
    def lr_fn(s):
        if s < c.warmup_steps: return s / max(1, c.warmup_steps)
        if s < he: return 1.0
        p = (s - he) / max(1, c.max_steps - he)
        return max(mr, 0.5 * (1.0 + math.cos(math.pi * p)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
    hist = {'step': [], 'train_ce': [], 'val_ce': [], 'train_acc': [], 'val_acc': [],
            'train_bpc': [], 'val_bpc': [], 'step_times': [], 'per_step_loss': []}
    model.train()
    t0 = time.time()
    smooth_loss = None
    pbar = tqdm(range(c.max_steps + 1), desc=label, unit='step')
    for step in pbar:
        if step % c.eval_interval == 0:
            r = estimate_loss(model, c)
            tr, vl = r['train'], r['val']
            hist['step'].append(step)
            hist['train_ce'].append(tr['ce']); hist['val_ce'].append(vl['ce'])
            hist['train_acc'].append(tr['acc']); hist['val_acc'].append(vl['acc'])
            hist['train_bpc'].append(tr['ce'] / math.log(2))
            hist['val_bpc'].append(vl['ce'] / math.log(2))
            vl_ppl = math.exp(min(vl['ce'], 20))
            tqdm.write(f"  [{label}] {step:5d} | Val CE:{vl['ce']:.3f} "
                       f"BPC:{vl['ce']/math.log(2):.3f} PPL:{vl_ppl:.1f} Acc:{vl['acc']:.1%}")
            save_hist(hist, label)
        if step >= c.max_steps:
            break
        st = time.time()
        batch = get_batch(train_ids, c)
        opt.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(device.type == 'cuda')):
            logits, _ = model(batch)
            tgt = batch[:, 1:]
            loss = F.cross_entropy(logits.reshape(-1, c.vocab_size), tgt.reshape(-1))
        if torch.isnan(loss):
            tqdm.write(f"  [{label}] NaN loss at step {step}, stopping early")
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        elapsed = time.time() - st
        hist['step_times'].append(elapsed)
        hist['per_step_loss'].append(loss.item())
        if smooth_loss is None: smooth_loss = loss.item()
        else: smooth_loss = 0.95 * smooth_loss + 0.05 * loss.item()
        ppl = math.exp(min(smooth_loss, 20))
        pbar.set_postfix(loss=f"{smooth_loss:.3f}", ppl=f"{ppl:.1f}",
                         bpc=f"{smooth_loss/math.log(2):.2f}",
                         lr=f"{sched.get_last_lr()[0]:.1e}", ms=f"{elapsed*1000:.0f}")
    pbar.close()
    torch.save({'step': step, 'model': model.state_dict(), 'hist': hist},
               os.path.join(RESULTS_DIR, f'{label}_final.pt'))
    el = time.time() - t0
    ms = np.mean(hist['step_times']) * 1000 if hist['step_times'] else 0
    final_ppl = math.exp(min(hist['val_ce'][-1], 20)) if hist['val_ce'] else float('nan')
    print(f"  {label} DONE: {el/60:.1f}min | BPC:{hist['val_bpc'][-1]:.3f} "
          f"PPL:{final_ppl:.1f} Acc:{hist['val_acc'][-1]:.1%} | {ms:.0f}ms/step")
    hist['avg_step_ms'] = ms; hist['n_params'] = count_params(model)
    save_hist(hist, label)
    return hist

## Re-initialize Models on GPU

The speed check moved models to CPU. Re-initialize fresh on GPU for training.

In [9]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

models = {}
# models['GPT+RoPE']      = SoftmaxModel(cfg, rope_mode='rope').to(device)
models['GPT+CurvBias']  = SoftmaxModel(cfg, rope_mode='curvbias').to(device)
# models['GLA+RoPE']      = GLAModel(cfg, rope_mode='rope').to(device)
models['GLA+CurvBias']  = GLAModel(cfg, rope_mode='curvbias').to(device)

print("Models re-initialized on device for training.")
for name, m in models.items():
    print(f"  {name:<18}: {count_params(m):,} params")

Models re-initialized on device for training.
  GPT+CurvBias      : 77,784,273 params
  GLA+CurvBias      : 77,870,417 params


In [ ]:
all_hist = {}
for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    all_hist[name] = train_model(model, cfg, label=name)
    # Save combined results after each model completes
    with open(os.path.join(RESULTS_DIR, 'all_hist.json'), 'w') as f:
        _json.dump({k: {kk: vv for kk, vv in v.items()
                        if not isinstance(vv, (float,)) or not math.isnan(vv)}
                    for k, v in all_hist.items()}, f, indent=1)
    print(f"  Results saved to {RESULTS_DIR}/ ({len(all_hist)} models complete)")
    # Free GPU memory for next model
    model.cpu()
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()


Training GPT+CurvBias...


GPT+CurvBias:   0%|          | 0/40001 [00:00<?, ?step/s]

  [GPT+CurvBias]     0 | Val CE:11.008 BPC:15.881 PPL:60358.4 Acc:0.0%
  [GPT+CurvBias]  5000 | Val CE:4.132 BPC:5.961 PPL:62.3 Acc:32.3%
  [GPT+CurvBias] 10000 | Val CE:3.689 BPC:5.322 PPL:40.0 Acc:36.6%
  [GPT+CurvBias] 15000 | Val CE:3.515 BPC:5.072 PPL:33.6 Acc:38.3%
  [GPT+CurvBias] 20000 | Val CE:3.409 BPC:4.918 PPL:30.2 Acc:39.4%
  [GPT+CurvBias] 25000 | Val CE:3.317 BPC:4.785 PPL:27.6 Acc:40.3%
  [GPT+CurvBias] 30000 | Val CE:3.265 BPC:4.711 PPL:26.2 Acc:41.0%
  [GPT+CurvBias] 35000 | Val CE:3.214 BPC:4.636 PPL:24.9 Acc:41.5%
  [GPT+CurvBias] 40000 | Val CE:3.200 BPC:4.616 PPL:24.5 Acc:41.6%
  GPT+CurvBias DONE: 194.0min | BPC:4.616 PPL:24.5 Acc:41.6% | 166ms/step
  Results saved to v6_results/ (1 models complete)

Training GLA+CurvBias...


GLA+CurvBias:   0%|          | 0/40001 [00:00<?, ?step/s]

  [GLA+CurvBias]     0 | Val CE:11.017 BPC:15.895 PPL:60919.3 Acc:0.0%


## Results

In [ ]:
colors = {
    'GPT+RoPE': 'gray', 'GPT+CurvBias': 'darkviolet',
    'GLA+RoPE': 'tab:blue', 'GLA+CurvBias': 'darkgreen',
}
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('V6 CurvBias Ablation (T=3072): Fixes A+B+G vs RoPE Baseline',
             fontsize=14, fontweight='bold')

for ax, key, title in [(axes[0,0], 'val_bpc', 'Val BPC'),
                        (axes[0,1], 'val_ce', 'Val Perplexity'),
                        (axes[0,2], 'val_acc', 'Val Accuracy %')]:
    for name, h in all_hist.items():
        if 'ppl' in title.lower():
            y = [math.exp(min(ce, 20)) for ce in h[key]]
        elif 'acc' in title.lower():
            y = [a * 100 for a in h[key]]
        else:
            y = h[key]
        ax.plot(h['step'], y, '-o', color=colors[name], label=name, markersize=3)
    ax.set_xlabel('Step'); ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

w = 100
for ax, use_ppl in [(axes[1,0], False), (axes[1,1], True)]:
    for name, h in all_hist.items():
        if len(h['per_step_loss']) > w:
            sm = np.convolve(h['per_step_loss'], np.ones(w)/w, mode='valid')
            y = [math.exp(min(x, 20)) for x in sm] if use_ppl else sm
            ax.plot(range(len(y)), y, '-', color=colors[name], label=name, alpha=0.8)
    ax.set_title(f'Step {"Perplexity" if use_ppl else "Loss"} (smooth {w})')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Summary table
ax = axes[1, 2]; ax.axis('off')
rows = []
for name, h in all_hist.items():
    ppl = math.exp(min(h['val_ce'][-1], 20))
    rows.append([name, f"{h['n_params']:,}", f"{h['val_bpc'][-1]:.3f}",
                 f"{ppl:.1f}", f"{h['val_acc'][-1]:.1%}", f"{h['avg_step_ms']:.0f}"])
t = ax.table(cellText=rows, colLabels=['Model', 'Params', 'BPC', 'PPL', 'Acc', 'ms/step'],
             loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.2, 1.6)
ax.set_title('Final Results', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('v6_ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 70)
print('V6 ABLATION RESULTS: Long-Sequence CurvBias (Fixes A+B+G)')
print('=' * 70)
for name, h in all_hist.items():
    ppl = math.exp(min(h['val_ce'][-1], 20))
    print(f"  {name:<18} BPC:{h['val_bpc'][-1]:.3f}  PPL:{ppl:.1f}  "
          f"Params:{h['n_params']:,}  {h['avg_step_ms']:.0f}ms/step")

## Gap Analysis

In [ ]:
def get_ppl(name):
    return math.exp(min(all_hist[name]['val_ce'][-1], 20))

ppls = {name: get_ppl(name) for name in all_hist}

print("\nKey comparisons (PPL, lower is better):")
print("=" * 60)

def ppl_cmp(a, b):
    if a in ppls and b in ppls:
        d = ppls[a] - ppls[b]
        print(f"  {b} vs {a}: {d:+.1f} ({d/ppls[a]*100:+.1f}%)")

print("\n--- CurvBias (fixed) vs RoPE baseline ---")
ppl_cmp('GPT+RoPE', 'GPT+CurvBias')
ppl_cmp('GLA+RoPE', 'GLA+CurvBias')

print("\n--- Cross-architecture (softmax vs linear) ---")
ppl_cmp('GPT+RoPE', 'GLA+RoPE')
ppl_cmp('GPT+CurvBias', 'GLA+CurvBias')

print("\n--- Ranking ---")
ranked = sorted(ppls.items(), key=lambda x: x[1])
for i, (name, ppl) in enumerate(ranked):
    print(f"  {i+1}. {name:<18} PPL: {ppl:.1f}")

print("\n--- Speed comparison ---")
for name, h in all_hist.items():
    print(f"  {name:<18}: {h['avg_step_ms']:.0f}ms/step  "
          f"overhead vs RoPE: {'+' if 'CurvBias' in name else ''}"
          f"{'N/A' if 'RoPE' in name else ''}")

# CurvBias overhead
if 'GPT+RoPE' in all_hist and 'GPT+CurvBias' in all_hist:
    base = all_hist['GPT+RoPE']['avg_step_ms']
    curv = all_hist['GPT+CurvBias']['avg_step_ms']
    print(f"\n  Softmax CurvBias overhead: {curv-base:+.0f}ms ({(curv-base)/base*100:+.1f}%)")
if 'GLA+RoPE' in all_hist and 'GLA+CurvBias' in all_hist:
    base = all_hist['GLA+RoPE']['avg_step_ms']
    curv = all_hist['GLA+CurvBias']['avg_step_ms']
    print(f"  Linear CurvBias overhead:  {curv-base:+.0f}ms ({(curv-base)/base*100:+.1f}%)")

## Diagnostics: Training Curve Analysis

Compare convergence speed and final performance. CurvBias should show:
1. Similar or better final PPL vs RoPE (content-aware bias helps)
2. Moderate speed overhead from circular distance GEMM (but much less than v5 cdist)
3. Stable training (no NaN from unbounded distances)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('V6 Training Diagnostics: CurvBias (A+B+G) vs RoPE at T=3072',
             fontsize=14, fontweight='bold')

# --- 1. Softmax PPL ---
ax = axes[0, 0]
for name in ['GPT+RoPE', 'GPT+CurvBias']:
    if name in all_hist:
        h = all_hist[name]
        ppl = [math.exp(min(ce, 20)) for ce in h['val_ce']]
        ax.plot(h['step'], ppl, '-o', color=colors[name], label=name, markersize=4)
ax.set_xlabel('Step'); ax.set_ylabel('Val PPL')
ax.set_title('Softmax Attention: Val Perplexity')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# --- 2. Linear PPL ---
ax = axes[0, 1]
for name in ['GLA+RoPE', 'GLA+CurvBias']:
    if name in all_hist:
        h = all_hist[name]
        ppl = [math.exp(min(ce, 20)) for ce in h['val_ce']]
        ax.plot(h['step'], ppl, '-o', color=colors[name], label=name, markersize=4)
ax.set_xlabel('Step'); ax.set_ylabel('Val PPL')
ax.set_title('Linear Attention: Val Perplexity')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# --- 3. Best PPL bar chart ---
ax = axes[0, 2]
names_ordered = ['GPT+RoPE', 'GPT+CurvBias', 'GLA+RoPE', 'GLA+CurvBias']
best_ppls = []
bar_colors = []
bar_names = []
for name in names_ordered:
    if name in all_hist:
        h = all_hist[name]
        best = min(math.exp(min(ce, 20)) for ce in h['val_ce'])
        best_ppls.append(best)
        bar_colors.append(colors[name])
        bar_names.append(name)
if best_ppls:
    bars = ax.barh(range(len(bar_names)), best_ppls, color=bar_colors, alpha=0.8)
    ax.set_yticks(range(len(bar_names)))
    ax.set_yticklabels(bar_names, fontsize=10)
    ax.set_xlabel('Best Val PPL (lower is better)')
    ax.set_title('Best PPL Achieved')
    for i, v in enumerate(best_ppls):
        ax.text(v + 0.5, i, f'{v:.1f}', va='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='x')

# --- 4. Accuracy (softmax) ---
ax = axes[1, 0]
for name in ['GPT+RoPE', 'GPT+CurvBias']:
    if name in all_hist:
        h = all_hist[name]
        ax.plot(h['step'], [a*100 for a in h['val_acc']], '-o',
                color=colors[name], label=name, markersize=4)
ax.set_xlabel('Step'); ax.set_ylabel('Val Accuracy %')
ax.set_title('Softmax: Accuracy')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# --- 5. Accuracy (linear) ---
ax = axes[1, 1]
for name in ['GLA+RoPE', 'GLA+CurvBias']:
    if name in all_hist:
        h = all_hist[name]
        ax.plot(h['step'], [a*100 for a in h['val_acc']], '-o',
                color=colors[name], label=name, markersize=4)
ax.set_xlabel('Step'); ax.set_ylabel('Val Accuracy %')
ax.set_title('Linear: Accuracy')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# --- 6. Summary table ---
ax = axes[1, 2]; ax.axis('off')
summary = []
for name in names_ordered:
    if name in all_hist:
        h = all_hist[name]
        final_ppl = math.exp(min(h['val_ce'][-1], 20))
        best_ppl = min(math.exp(min(ce, 20)) for ce in h['val_ce'])
        summary.append([name,
                        f"{h.get('n_params','?'):,}" if isinstance(h.get('n_params'), int) else '?',
                        f"{final_ppl:.1f}", f"{best_ppl:.1f}",
                        f"{h['val_acc'][-1]:.1%}", f"{h.get('avg_step_ms',0):.0f}"])
if summary:
    t = ax.table(cellText=summary,
                 colLabels=['Model', 'Params', 'Final PPL', 'Best PPL', 'Acc', 'ms/step'],
                 loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.2, 1.6)
ax.set_title('Summary', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('v6_ablation_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()